In [ ]:
!pip install -q unsloth wandb rouge-score
!pip install --no-deps -q trl peft accelerate bitsandbytes
!pip install -q 'anthropic[bedrock]' boto3

from google.colab import drive
drive.mount('/content/drive')

import os
from google.colab import userdata

# ── AWS Bedrock creds (Colab Secrets) ──────────────────────────────────────
os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = "us-west-2"

# ── paths (v4: trace distillation on option-2 reconciliation dataset) ────
DATA_PATH    = '/content/train_dataset_v4_traces_o2.jsonl'
OUTPUT_DIR   = '/content/drive/MyDrive/sft/outputs-sft-v4'
ADAPTER_DIR  = '/content/drive/MyDrive/sft/code-reviewer-lora-v4-traces'
MERGED_DIR   = '/content/drive/MyDrive/sft/sft-v4-merged-for-eval'

os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.exists(DATA_PATH), (
    f"Missing {DATA_PATH}\n"
    "Upload train_dataset_v4_traces_o2.jsonl from your Mac to /content/ (this session) "
    "or to Google Drive under /MyDrive/sft/."
)

# ── W&B setup ──────────────────────────────────────────────────────────────
# On first run Colab will prompt for your W&B API key. It persists for the session.
import wandb
wandb.login()   # paste key from https://wandb.ai/authorize when prompted

WANDB_PROJECT = "code-reviewer-sft"
WANDB_RUN     = "sft-v4-traces"

# Hyperparameters are auto-logged by SFTTrainer via report_to="wandb"; keeping
# them out of this config avoids drift with the values hardcoded in Cell 3.
wandb_run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN,
    config={
        "phase":     "sft-v4-traces",
        "data_path": DATA_PATH,
    },
)

print(f"✓ Data:    {DATA_PATH}")
print(f"✓ Outputs: {OUTPUT_DIR}")
print(f"✓ Adapter: {ADAPTER_DIR}")
print(f"✓ Merged:  {MERGED_DIR}")
print(f"✓ W&B run: {wandb_run.url if wandb_run else 'n/a'}")
print(f"✓ Bedrock region: {os.environ['AWS_DEFAULT_REGION']}")


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192   # enough for long diffs + <think>+<review> trace target
dtype = None
load_in_4bit = False    # A100 80GB has plenty of VRAM for full precision

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,            # v4: was 32 — 2× alpha gives more headroom for trace SFT (per MelcotCR recipe)
    lora_dropout=0.05,        # Phase 2: was 0 — known Unsloth footgun for multi-epoch runs
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("✅ Model loaded")


In [ ]:
import json, random
from datasets import Dataset

# ── Leakage guard ──────────────────────────────────────────────────────────
# eval.ipynb Cell 2 picks the eval set as `random.shuffle(range(len(data)))[:200]`
# with seed 42 on train_dataset_v3.jsonl (which has 12876 records). We reproduce
# that exact shuffle here to get the same 200 indices, then exclude them from
# SFT training. Every v4 trace record stores `_idx` = its index into the v3
# source file, so the dedupe is a direct set-membership check.
random.seed(42)
_TOTAL_V3 = 12876
_indices = list(range(_TOTAL_V3))
random.shuffle(_indices)
EVAL_INDICES = set(_indices[:200])

# ── Load + filter ─────────────────────────────────────────────────────────
data = []
skipped_non_trace = 0
skipped_eval = 0
with open(DATA_PATH, 'r') as f:
    for line in f:
        r = json.loads(line)
        if r.get('trace_status') != 'trace_generated':
            skipped_non_trace += 1                # drop filtered/junk/incomplete
            continue
        if r.get('_idx') in EVAL_INDICES:
            skipped_eval += 1                     # data-leakage guard
            continue
        data.append(r)

print(f"Loaded {len(data)} SFT-ready samples")
print(f"  skipped {skipped_non_trace} non-trace records (filtered/junk/incomplete)")
print(f"  skipped {skipped_eval} eval-set records (leakage guard)")

def format_prompt(sample):
    system_msg = "You are a Senior Software Engineer reviewing code changes. Provide clear, actionable feedback."

    # Phase 2 fixes carried over: 3-backtick fence, 3000-char truncation matches eval + GRPO
    user_msg = f"""Review the following code diff and provide feedback:
```diff
{sample['input'][:3000]}
```"""

    # v4: sample['output'] is already <think>{call1}</think><review>{call2_final}</review>
    # — produced by generate_traces_gemini.py. No reshaping needed.
    assistant_msg = sample['output']

    text = f"""<|im_start|>system
{system_msg}<|im_end|>
<|im_start|>user
{user_msg}<|im_end|>
<|im_start|>assistant
{assistant_msg}<|im_end|>"""

    return {"text": text}


dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompt)
dataset = dataset.train_test_split(test_size=0.05, seed=42)
print(f"Train: {len(dataset['train'])}, Val: {len(dataset['test'])}")


In [5]:
# ===== CELL 3: Training (A100 80GB) =====
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=4,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,       # effective batch 16
        warmup_steps=100,                    # Phase 2: was 50 — safer with lower LR
        num_train_epochs=2,                  # Phase 2: was 3 — overfitting signature
        learning_rate=1e-4,                  # Phase 2: was 2e-4 — pair with dropout
        fp16=False,
        bf16=True,
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir=OUTPUT_DIR,               # Phase 2: Drive path from Cell 0
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
        report_to="wandb",                   # auto-logs loss, eval_loss, lr, grad_norm
        run_name=WANDB_RUN,
    ),
)

print("🚀 Starting training...")
trainer_stats = trainer.train()
print("✅ Training complete")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/12232 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/644 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,232 | Num Epochs = 2 | Total steps = 1,530
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 80,740,352 of 7,696,356,864 (1.05% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
100,0.971459,0.949495
200,0.894278,0.894196
300,0.833442,0.865754
400,0.796800,0.845540
500,0.828528,0.831090
600,0.774569,0.816586
700,0.769117,0.806337
800,0.691138,0.801215
900,0.717661,0.794417
1000,0.690999,0.788883


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

✅ Training complete


In [6]:
# ===== CELL 4: Save LoRA adapter + merged model to Drive =====
# Adapter (~300 MB) — reuse for future training runs or GRPO
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✓ Adapter saved: {ADAPTER_DIR}")

# Merged model (~14 GB) — use this for eval/inference; fuses LoRA into base weights
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)
print(f"✓ Merged model saved: {MERGED_DIR}")
print("\nSFT v3 complete. Both artifacts persist on Drive.")
print("Next steps:")
print("  - Quick eval: run cells 9-10 (may need MERGED_DIR path update)")
print("  - Full eval + GRPO: separate notebooks, use MERGED_DIR as base")

✓ Adapter saved: /content/drive/MyDrive/sft/code-reviewer-lora-v3
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 4 files from cache to `/content/drive/MyDrive/sft/sft-merged-for-eval`:   0%|          | 0/4 [00:00<?, ?it/s]
Unsloth: Copying 4 files from cache to `/content/drive/MyDrive/sft/sft-merged-for-eval`:  25%|██▌       | 1/4 [00:11<00:33, 11.30s/it]
Unsloth: Copying 4 files from cache to `/content/drive/MyDrive/sft/sft-merged-for-eval`:  50%|█████     | 2/4 [00:31<00:33, 16.64s/it]
Unsloth: Copying 4 files from cache to `/content/drive/MyDrive/sft/sft-merged-for-eval`:  75%|███████▌  | 3/4 [00:49<00:17, 17.01s/it]
Unsloth: Copying 4 files from cache to `/content/drive/MyDrive/sft/sft-merged-for-eval`: 100%|██████████| 4/4 [00:57<00:00, 14.42s/it]


Successfully copied all 4 files from cache to `/content/drive/MyDrive/sft/sft-merged-for-eval`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 9430.70it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:11<00:00, 17.84s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/sft/sft-merged-for-eval`
✓ Merged model saved: /content/drive/MyDrive/sft/sft-merged-for-eval

SFT v3 complete. Both artifacts persist on Drive.
Next steps:
  - Quick eval: run cells 9-10 (may need MERGED_DIR path update)
  - Full eval + GRPO: separate notebooks, use MERGED_DIR as base


In [ ]:
# ===== CELL 5: Sanity test — 3 diffs (security / bug / style) =====
FastLanguageModel.for_inference(model)

test_diffs = [
    # Security issue — hardcoded password
    """@@ -12,7 +12,7 @@ def connect(self):
-        password = os.environ.get('DB_PASS')
+        password = "admin123"
         return db.connect(password)""",

    # Bug — .first() removal (the failure pattern we targeted in Phase 1.4)
    """@@ -5,7 +5,7 @@ def get_user(user_id):
-        user = db.query(User).filter(User.id == user_id).first()
+        user = db.query(User).filter(User.id == user_id)
         return user.name""",

    # Style — bad renames
    """@@ -1,5 +1,5 @@
-def calculateTotalPrice(items):
+def calc(i):
     total = 0
-    for item in items:
+    for x in i:
         total += x.price""",
]

# Deterministic decoding so the sanity test is reproducible across runs.
# v4: bumped max_new_tokens 256 → 1024 because traces average ~3500 chars
# (≈1000 tokens) — 256 truncated mid-<think> on the first v4 run.
# return_dict=True so we also get attention_mask, silencing the "pad==eos"
# warning and ensuring reliable generation.
for i, diff in enumerate(test_diffs, 1):
    messages = [
        {"role": "system", "content": "You are a Senior Software Engineer reviewing code changes. Provide clear, actionable feedback."},
        {"role": "user",   "content": f"Review this code diff:\n\n```diff\n{diff}\n```"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=1024,
        temperature=0,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    review = response.split("assistant")[-1].strip()

    print(f"\n{'='*50}")
    print(f"TEST {i}")
    print(f"{'='*50}")
    print(f"DIFF:\n{diff}")
    print(f"\nMODEL REVIEW:\n{review}")


In [ ]:
# ===== CELL 6: Generate eval predictions — Base vs SFT =====
import json, random, torch, gc
from unsloth import FastLanguageModel

# Free the trainer + LoRA-wrapped model before loading eval models.
# A100 80GB has headroom, but smaller GPUs would OOM on the second from_pretrained.
try:
    del trainer, model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# Load 50 samples
data = []
with open(DATA_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

random.seed(42)
synthetic = [r for r in data if r.get('repo', '').startswith('synthetic/')]
regular   = [r for r in data if not r.get('repo', '').startswith('synthetic/')]
random.shuffle(regular)
quick_samples = (synthetic[:10] + regular[:40])[:50]
random.shuffle(quick_samples)

SYSTEM_MSG = "You are a Senior Software Engineer reviewing code changes. Provide clear, actionable feedback."
BASE_MODEL = "unsloth/Qwen2.5-Coder-7B-Instruct"
SFT_MERGED = MERGED_DIR

def run_eval(model_path, samples):
    m, tok = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=4096,
        dtype=torch.bfloat16,
        load_in_4bit=False,
    )
    FastLanguageModel.for_inference(m)
    preds, refs, diffs = [], [], []
    for i, s in enumerate(samples):
        diff = s['input'][:3000]
        msgs = [
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user",   "content": f"Review this code diff:\n\n```diff\n{diff}\n```"},
        ]
        inputs = tok.apply_chat_template(
            msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")
        with torch.no_grad():
            out = m.generate(
                input_ids=inputs, max_new_tokens=256,
                temperature=0, do_sample=False,
                pad_token_id=tok.eos_token_id,
            )
        text = tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()
        preds.append(text); refs.append(s['output']); diffs.append(diff)
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(samples)}")
    del m, tok
    gc.collect(); torch.cuda.empty_cache()
    return preds, refs, diffs

print(f"Quick eval: {len(quick_samples)} samples\n")
print("Loading base model...")
base_preds, base_refs, base_diffs = run_eval(BASE_MODEL, quick_samples)
print("\nLoading SFT model...")
sft_preds,  sft_refs,  sft_diffs  = run_eval(SFT_MERGED, quick_samples)
print("\nDone — run Cell 7 for results table")


In [ ]:
# ===== CELL 7: QUICK EVAL RESULTS — Base vs SFT =====
import re, numpy as np
from rouge_score import rouge_scorer
from concurrent.futures import ThreadPoolExecutor, as_completed
import anthropic

_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
_client = anthropic.AnthropicBedrock(aws_region="us-west-2")
HAIKU_MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
_stop   = {'self','return','import','class','True','False','None','with','from',
           'def','the','and','for','not','this','that','list','dict','str','int'}
_haiku_errors = 0

def haiku_score_quick(review, diff, ref):
    global _haiku_errors
    try:
        r = _client.messages.create(
            model=HAIKU_MODEL, max_tokens=4,
            messages=[{"role": "user", "content": (
                "Rate this code review 0 to 10.\n\n"
                f"CODE DIFF:\n{diff[:2000]}\n\n"
                f"REFERENCE:\n{ref[:500]}\n\n"
                f"REVIEW:\n{review[:500]}\n\n"
                "Score anchors:\n"
                "  5 = average — correct but generic, could apply to any diff.\n"
                "  7+ = requires specific line references, variable names, or function names from the diff.\n"
                "  3 or below = vague advice that could be written without reading the diff.\n"
                "Reply ONLY with a single integer 0-10."
            )}]
        )
        m = re.search(r'\d+', r.content[0].text)
        return min(max(int(m.group()), 0), 10) if m else 5
    except Exception as e:
        # Track failures — a bare except previously made auth/rate-limit/model-ID issues
        # look like a clean 5.0 mean. Surface the count and the first few errors.
        _haiku_errors += 1
        if _haiku_errors <= 3:
            print(f"[WARN] Haiku scoring failed ({_haiku_errors}): {e}")
        return 5

def batch_haiku(preds, diffs, refs):
    scores = [5] * len(preds)
    with ThreadPoolExecutor(max_workers=16) as ex:
        futs = {ex.submit(haiku_score_quick, p, d, r): i
                for i, (p, d, r) in enumerate(zip(preds, diffs, refs))}
        for f in as_completed(futs):
            scores[futs[f]] = f.result()
    return scores

def _detect(preds, diffs, min_len):
    """Fraction of predictions that mention any identifier (≥min_len chars) from the diff's +/- lines."""
    pattern = re.compile(rf'\b[a-zA-Z_][a-zA-Z0-9_]{{{min_len-1},}}\b')
    n = 0
    for pred, diff in zip(preds, diffs):
        changed = ' '.join(l[1:] for l in diff.split('\n') if l.startswith(('+', '-')))
        ids = set(pattern.findall(changed)) - _stop
        if any(i in pred for i in ids):
            n += 1
    return n / len(preds) if preds else 0

def issue_detection(preds, diffs):  return _detect(preds, diffs, min_len=3)
def specificity_rate(preds, diffs): return _detect(preds, diffs, min_len=4)

def rougeL_mean(preds, refs):
    return float(np.mean([_scorer.score(r, p)['rougeL'].fmeasure for p, r in zip(preds, refs)]))

print("Scoring with Haiku (100 calls, ~$0.01)...")
base_scores = batch_haiku(base_preds, base_diffs, base_refs)
sft_scores  = batch_haiku(sft_preds,  sft_diffs,  sft_refs)
if _haiku_errors:
    print(f"⚠ {_haiku_errors} Haiku scoring errors — means may be biased toward the 5 default")

results = {
    "Base": dict(
        haiku_mean      = float(np.mean(base_scores)),
        pass_at_1       = sum(s >= 7 for s in base_scores) / len(base_scores),
        issue_detection = issue_detection(base_preds, base_diffs),
        specificity     = specificity_rate(base_preds, base_diffs),
        rougeL          = rougeL_mean(base_preds, base_refs),
    ),
    "SFT": dict(
        haiku_mean      = float(np.mean(sft_scores)),
        pass_at_1       = sum(s >= 7 for s in sft_scores) / len(sft_scores),
        issue_detection = issue_detection(sft_preds, sft_diffs),
        specificity     = specificity_rate(sft_preds, sft_diffs),
        rougeL          = rougeL_mean(sft_preds, sft_refs),
    ),
}

print(f"\n{'Metric':<20} {'Base':>10} {'SFT':>10} {'Delta':>10}")
print("=" * 52)
for m in ["haiku_mean", "pass_at_1", "issue_detection", "specificity", "rougeL"]:
    b, s = results["Base"][m], results["SFT"][m]
    arrow = "↑" if s > b else ("↓" if s < b else "=")
    print(f"{m:<20} {b:>10.4f} {s:>10.4f} {arrow}{s-b:>+9.4f}")
print("=" * 52)
print("\nTarget: issue_detection SFT ≥ 0.87 (base is ~0.87, old SFT regressed to 0.80)")

# ── Log to W&B ─────────────────────────────────────────────────────────────
import wandb
if wandb.run is not None:
    flat = {}
    for model, metrics in results.items():
        for m, v in metrics.items():
            flat[f"quick_eval/{model.lower()}/{m}"] = v
    wandb.log(flat)
    print(f"\n✓ Quick eval metrics logged to {wandb.run.url}")

# Failure pattern spot-check
print("\n── Failure Pattern Spot-Check ──")
FAILURE_KW = ['.first()', '"true"', '"false"', 'MAX_', 'TIMEOUT', 'RETRY']
found = 0
for i, (diff, bp, sp) in enumerate(zip(base_diffs, base_preds, sft_preds)):
    for kw in FAILURE_KW:
        if kw.lower() in diff.lower():
            print(f"\nSample {i} — diff contains '{kw}':")
            print(f"  BASE: {bp[:200]}")
            print(f"  SFT:  {sp[:200]}")
            found += 1
            break
if found == 0:
    print("  (no failure-pattern samples in this batch — re-run with larger quick_samples)")

# Close run at end of quick-eval
if wandb.run is not None:
    wandb.finish()
    print("\n✓ W&B run closed")
